# VALOR-32K Online Builder (Batched + Incremental Save)

This notebook downloads small **batched** subsets of VALOR-32K from YouTube, extracts a middle video frame and audio segment **on the fly**, and saves them
to Parquet files in **small shards** so that progress is continually saved.

Design goals:
- Start with a **small batch** so you can already begin Phase-2 training.
- Run later batches independently and **merge all shards** at the end.
- Avoid keeping any video clips on disk (use temp files only).
- Use **thread-based parallelism** (safer on macOS + Jupyter).

Each processed example is stored as:

| column      | type   |
|------------|--------|
| `video_id` | str    |
| `yt_id`    | str    |
| `start`    | float  |
| `end`      | float  |
| `caption`  | str    |
| `image_jpeg` | bytes (JPEG) |
| `audio_wav`  | bytes (WAV, 16kHz mono) |

Shards will be written as files named like:

`valor32k_train_batch000_shard000.parquet`, `valor32k_train_batch000_shard001.parquet`, ...

You can later merge shards across batches into a single big Parquet if you want.

In [1]:
# ============================================================
# Config: paths, batch control, and shard settings
# ============================================================
from pathlib import Path

# Root of your project (edit if needed)
ROOT_DIR = Path.cwd()

# Path to VALOR-32K annotations JSON (train split example)
ANNOTATIONS_JSON = ROOT_DIR / "data" / "valor_32k" / "desc_train.json"  # <-- EDIT

# Output directory for Parquet shards
OUTPUT_DIR = ROOT_DIR / "data" / "alignment_subsets" / "valor32k_train_shards"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Batching config
BATCH_SIZE = 1000   # small initial batch
BATCH_INDEX = 0     # 0 = first batch

# Shard config (incremental save)
SHARD_SIZE = 200    # rows per shard

# Worker threads
NUM_WORKERS = 4

# Image resize
IMG_WIDTH = 224
IMG_HEIGHT = 224

print("ROOT_DIR        :", ROOT_DIR)
print("ANNOTATIONS_JSON:", ANNOTATIONS_JSON)
print("OUTPUT_DIR      :", OUTPUT_DIR)
print("BATCH_SIZE      :", BATCH_SIZE)
print("BATCH_INDEX     :", BATCH_INDEX)

ROOT_DIR        : /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Edge Assistant/code_base/v2_code_base
ANNOTATIONS_JSON: /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Edge Assistant/code_base/v2_code_base/data/valor_32k/desc_train.json
OUTPUT_DIR      : /Users/vedaangchopra/all_data/complete_technical_work/all_projects_implemented/Edge Assistant/code_base/v2_code_base/data/alignment_subsets/valor32k_train_shards
BATCH_SIZE      : 1000
BATCH_INDEX     : 0


In [2]:
# ============================================================
# Imports & environment checks
# ============================================================
import os
import json
import tempfile
import subprocess
import time
from functools import partial
from typing import Any, Dict, Optional, Tuple

from multiprocessing.dummy import Pool as ThreadPool

import cv2
import pandas as pd
from tqdm.auto import tqdm

print("Python version:", os.sys.version)

def _check_binary(name: str) -> bool:
    from shutil import which
    return which(name) is not None

print("yt-dlp available:", _check_binary("yt-dlp"))
print("ffmpeg available:", _check_binary("ffmpeg"))

if not _check_binary("yt-dlp"):
    print("⚠️ yt-dlp not found in PATH. Install it before running.")
if not _check_binary("ffmpeg"):
    print("⚠️ ffmpeg not found in PATH. Install it before running.")

Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:07:49) [Clang 20.1.8 ]
yt-dlp available: True
ffmpeg available: True


In [3]:
# ============================================================
# Helper functions
# ============================================================
def parse_video_id(video_id: str) -> Tuple[str, float, float]:
    parts = video_id.rsplit("_", 2)
    if len(parts) != 3:
        raise ValueError(f"Unexpected video_id format: {video_id}")
    yt_id, start_str, end_str = parts
    return yt_id, float(start_str), float(end_str)

FAILED_LOG = OUTPUT_DIR / "valor32k_failed_downloads.txt"
FAILED_LOG.parent.mkdir(parents=True, exist_ok=True)

def download_clip_to_temp(yt_id: str, start: float, end: float) -> Optional[Path]:
    url = f"https://www.youtube.com/watch?v={yt_id}"
    duration = max(0.0, end - start)
    if duration <= 0:
        print(f"[ERROR] Invalid duration {duration} for {yt_id}")
        return None

    tmp_dir = Path(tempfile.mkdtemp(prefix="valor_clip_"))
    out_path = tmp_dir / "clip.mp4"
    section = f"*{start}-{end}"

    cmd = [
        "yt-dlp",
        "-f", "mp4",
        "--download-sections", section,
        "-o", str(out_path),
        url,
    ]

    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True,
        )
    except subprocess.CalledProcessError as e:
        with FAILED_LOG.open("a", encoding="utf-8") as f:
            f.write(f"{yt_id},{start},{end}\n")
        print(f"[ERROR] yt-dlp failed for {yt_id}:")
        print("STDOUT:\n", e.stdout)
        print("STDERR:\n", e.stderr)
        try:
            if out_path.exists():
                out_path.unlink()
            tmp_dir.rmdir()
        except Exception:
            pass
        return None

    if not out_path.exists():
        print(f"[ERROR] Clip file not created for {yt_id}")
        try:
            tmp_dir.rmdir()
        except Exception:
            pass
        return None
    return out_path


def choose_frame_indices(
    frame_count: int,
    fps: float,
    min_frames: int = 1,
    max_frames: int = 5,
) -> list[int]:
    """
    Decide which frame indices to sample based on clip duration.
    Returns a list of integer frame indices (0-based).
    """
    if frame_count <= 0:
        return []

    # Fallback if fps is broken
    if fps is None or fps <= 0:
        fps = 25.0  # default guess

    duration = frame_count / fps

    # Heuristic: number of frames vs duration
    if duration < 2.0:
        n_frames = 1
    elif duration < 5.0:
        n_frames = 2
    elif duration < 12.0:
        n_frames = 3
    else:
        n_frames = 4

    n_frames = max(min_frames, min(max_frames, n_frames))

    # Evenly space indices across the clip, but avoid the very first/last frames
    # (which are often black/transition frames)
    import numpy as np

    # sample in [0.1, 0.9] of the clip to avoid edges
    positions = np.linspace(0.1, 0.9, n_frames)
    indices = [int(p * (frame_count - 1)) for p in positions]

    # ensure uniqueness and valid bounds
    indices = sorted(set(max(0, min(frame_count - 1, idx)) for idx in indices))
    return indices


def extract_adaptive_frames(
    video_path: Path,
    resize: Optional[Tuple[int, int]] = None,
) -> list[bytes]:
    """
    Extract an adaptive number of frames depending on video length.
    Returns a list of JPEG bytes (can be length 1–5 typically).
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))

    if frame_count <= 0:
        cap.release()
        return []

    indices = choose_frame_indices(frame_count, fps)
    frames_bytes: list[bytes] = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok or frame is None:
            continue

        if resize is not None:
            w, h = resize
            frame = cv2.resize(frame, (w, h), interpolation=cv2.INTER_AREA)

        success, buf = cv2.imencode(".jpg", frame)
        if success:
            frames_bytes.append(buf.tobytes())

    cap.release()
    return frames_bytes

def extract_middle_frame(video_path: Path, resize: Optional[Tuple[int, int]] = None) -> Optional[bytes]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return None
    middle_idx = frame_count // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, middle_idx)
    ok, frame = cap.read()
    cap.release()
    if not ok or frame is None:
        return None
    if resize is not None:
        w, h = resize
        frame = cv2.resize(frame, (w, h), interpolation=cv2.INTER_AREA)
    success, buf = cv2.imencode(".jpg", frame)
    if not success:
        return None
    return buf.tobytes()

def extract_audio_wav(video_path: Path, sr: int = 16000) -> Optional[bytes]:
    cmd = [
        "ffmpeg", "-i", str(video_path),
        "-vn", "-ac", "1", "-ar", str(sr),
        "-f", "wav", "-loglevel", "quiet", "pipe:1",
    ]
    try:
        proc = subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        return proc.stdout
    except subprocess.CalledProcessError:
        return None


In [4]:
# ============================================================
# Worker function
# ============================================================
def process_annotation(entry: Dict[str, Any], resize: Optional[Tuple[int, int]] = None) -> Optional[Dict[str, Any]]:
    video_id = entry.get("video_id")
    caption = entry.get("desc") or entry.get("caption") or ""
    if not video_id:
        print(f"[WARN] Missing video_id in entry: {entry}")
        return None
    try:
        yt_id, start, end = parse_video_id(video_id)
    except Exception as e:
        print(f"[ERROR] Failed to parse video_id={video_id}: {e}")
        return None
    t0 = time.time()
    print(f"\n--- Processing {video_id} ---")
    print(f"[INFO] yt_id={yt_id}  start={start}  end={end}")
    print("[INFO] Downloading clip segment...")
    clip_path = download_clip_to_temp(yt_id, start, end)
    if clip_path is None:
        print(f"[ERROR] Failed to download segment for {video_id}")
        return None
    print(f"[OK] Downloaded to: {clip_path}")
    try:
        # print("[INFO] Extracting middle frame...")
        # frame_bytes = extract_middle_frame(clip_path, resize=resize)
        # if frame_bytes is None:
        #     print("[ERROR] Failed to extract middle frame")
        #     raise RuntimeError("frame extraction failure")
        
        print("[INFO] Extracting frames (adaptive)...")
        frame_bytes_list = extract_adaptive_frames(clip_path, resize=resize)
        if not frame_bytes_list:
            print("[ERROR] Failed to extract any frames")
            raise RuntimeError("frame extraction failure")
        print(f"[OK] Extracted {len(frame_bytes_list)} frame(s)")

        print("[OK] Frame extracted")
        print("[INFO] Extracting audio wav...")
        audio_bytes = extract_audio_wav(clip_path, sr=16000)
        if audio_bytes is None:
            print("[ERROR] Failed to extract audio")
            raise RuntimeError("audio extraction failure")
        print(f"[OK] Audio extracted: {len(audio_bytes)} bytes")
    except Exception as e:
        print(f"[ERROR] Extraction failed for {video_id}: {e}")
        try:
            tmp_dir = clip_path.parent
            if clip_path.exists():
                clip_path.unlink()
            tmp_dir.rmdir()
        except Exception:
            pass
        return None
    print("[INFO] Cleaning up temporary files...")
    try:
        tmp_dir = clip_path.parent
        if clip_path.exists():
            clip_path.unlink()
        tmp_dir.rmdir()
        print("[OK] Cleanup complete")
    except Exception as e:
        print(f"[WARN] Cleanup issue: {e}")
    dt = time.time() - t0
    print(f"[SUCCESS] {video_id} processed in {dt:.2f}s")
    return {
    "video_id": video_id,
    "yt_id": yt_id,
    "start": start,
    "end": end,
    "caption": caption,
    "image_jpegs": frame_bytes_list,   # list[bytes]
    "audio_wav": audio_bytes,
}


In [5]:
# # ============================================================
# # Main: batched, sharded processing
# # ============================================================
# assert ANNOTATIONS_JSON.is_file(), f"Annotations JSON not found: {ANNOTATIONS_JSON}"
# with ANNOTATIONS_JSON.open("r", encoding="utf-8") as f:
#     all_annotations = json.load(f)
# if not isinstance(all_annotations, list):
#     raise ValueError("Annotations JSON must be a list of dicts.")
# total = len(all_annotations)
# print(f"Total annotations in file: {total}")
# start_idx = BATCH_INDEX * BATCH_SIZE
# end_idx = min(start_idx + BATCH_SIZE, total)
# if start_idx >= total:
#     raise ValueError(
#         f"BATCH_INDEX {BATCH_INDEX} with BATCH_SIZE {BATCH_SIZE} starts at {start_idx}, "
#         f"but dataset only has {total} entries."
#     )
# annotations = all_annotations[start_idx:end_idx]
# print(f"Processing batch {BATCH_INDEX} → indices [{start_idx}:{end_idx})")
# print(f"Batch size: {len(annotations)}")

In [6]:

# resize = (IMG_WIDTH, IMG_HEIGHT) if IMG_WIDTH > 0 and IMG_HEIGHT > 0 else None
# results_buffer = []
# num_success = 0
# num_failed = 0
# shard_index = 0

# def flush_shard(buffer, shard_index: int):
#     if not buffer:
#         return shard_index
#     df = pd.DataFrame(buffer)
#     out_path = OUTPUT_DIR / f"valor32k_train_batch{BATCH_INDEX:03d}_shard{shard_index:03d}.parquet"
#     df.to_parquet(out_path, index=False)
#     print(f"\n💾 Saved shard {shard_index} with {len(df)} rows → {out_path}")
#     return shard_index + 1


In [7]:
# if NUM_WORKERS <= 1:
#     for entry in tqdm(annotations, desc="Processing VALOR entries (single thread)"):
#         row = process_annotation(entry, resize=resize)
#         if row is None:
#             num_failed += 1
#         else:
#             num_success += 1
#             results_buffer.append(row)
#             if len(results_buffer) >= SHARD_SIZE:
#                 shard_index = flush_shard(results_buffer, shard_index)
#                 results_buffer = []
# else:
#     worker = partial(process_annotation, resize=resize)
#     with ThreadPool(processes=NUM_WORKERS) as pool:
#         for row in tqdm(
#             pool.imap_unordered(worker, annotations, chunksize=4),
#             total=len(annotations),
#             desc="Processing VALOR entries (threads)",
#         ):
#             if row is None:
#                 num_failed += 1
#             else:
#                 num_success += 1
#                 results_buffer.append(row)
#                 if len(results_buffer) >= SHARD_SIZE:
#                     shard_index = flush_shard(results_buffer, shard_index)
#                     results_buffer = []

# if results_buffer:
#     shard_index = flush_shard(results_buffer, shard_index)
#     results_buffer = []
# print("\n✅ Finished batch.")
# print(f"   Successful rows: {num_success}")
# print(f"   Failed/missed  : {num_failed}")
# print(f"   Shards written : {shard_index}")
# print(f"   Shard directory: {OUTPUT_DIR}")

In [11]:
# ============================================================
# Main: batched, sharded processing over multiple batches
# ============================================================
assert ANNOTATIONS_JSON.is_file(), f"Annotations JSON not found: {ANNOTATIONS_JSON}"
with ANNOTATIONS_JSON.open("r", encoding="utf-8") as f:
    all_annotations = json.load(f)
if not isinstance(all_annotations, list):
    raise ValueError("Annotations JSON must be a list of dicts.")
total = len(all_annotations)
print(f"Total annotations in file: {total}")

# define which batches to run in this execution
START_BATCH = BATCH_INDEX         # e.g. 0
END_BATCH   = 26     # e.g. 32 if you want all in one go

resize = (IMG_WIDTH, IMG_HEIGHT) if IMG_WIDTH > 0 and IMG_HEIGHT > 0 else None

global_success = 0
global_failed = 0
global_shards = 0


Total annotations in file: 25327


In [12]:

for current_batch in range(START_BATCH, END_BATCH):
    print(f"\n================ BATCH {current_batch} ================\n")

    start_idx = current_batch * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, total)
    if start_idx >= total:
        print(
            f"[WARN] BATCH_INDEX {current_batch} with BATCH_SIZE {BATCH_SIZE} "
            f"starts at {start_idx}, but dataset only has {total} entries. Stopping."
        )
        break

    annotations = all_annotations[start_idx:end_idx]
    print(f"Processing batch {current_batch} → indices [{start_idx}:{end_idx})")
    print(f"Batch size: {len(annotations)}")

    results_buffer = []
    num_success = 0
    num_failed = 0
    shard_index = 0

    def flush_shard(buffer, shard_index: int):
        if not buffer:
            return shard_index
        df = pd.DataFrame(buffer)
        out_path = OUTPUT_DIR / f"valor32k_train_batch{current_batch:03d}_shard{shard_index:03d}.parquet"
        df.to_parquet(out_path, index=False)
        print(f"\n💾 Saved shard {shard_index} with {len(df)} rows → {out_path}")
        return shard_index + 1

    if NUM_WORKERS <= 1:
        for entry in tqdm(annotations, desc=f"Batch {current_batch} (single thread)"):
            row = process_annotation(entry, resize=resize)
            if row is None:
                num_failed += 1
            else:
                num_success += 1
                results_buffer.append(row)
                if len(results_buffer) >= SHARD_SIZE:
                    shard_index = flush_shard(results_buffer, shard_index)
                    results_buffer = []
    else:
        worker = partial(process_annotation, resize=resize)
        with ThreadPool(processes=NUM_WORKERS) as pool:
            for row in tqdm(
                pool.imap_unordered(worker, annotations, chunksize=4),
                total=len(annotations),
                desc=f"Batch {current_batch} (threads)",
            ):
                if row is None:
                    num_failed += 1
                else:
                    num_success += 1
                    results_buffer.append(row)
                    if len(results_buffer) >= SHARD_SIZE:
                        shard_index = flush_shard(results_buffer, shard_index)
                        results_buffer = []

    if results_buffer:
        shard_index = flush_shard(results_buffer, shard_index)
        results_buffer = []

    print("\n✅ Finished batch", current_batch)
    print(f"   Successful rows: {num_success}")
    print(f"   Failed/missed  : {num_failed}")
    print(f"   Shards written : {shard_index}")

    global_success += num_success
    global_failed += num_failed
    global_shards += shard_index

print("\n🔚 All requested batches done.")
print(f"   Total successful rows: {global_success}")
print(f"   Total failed/missed  : {global_failed}")
print(f"   Total shards written : {global_shards}")
print(f"   Shard directory      : {OUTPUT_DIR}")



================ BATCH 0 ================

Processing batch 0 → indices [0:1000)
Batch size: 1000

--- Processing x-2Abohj8VY_30.000_40.000 ---
[INFO] yt_id=x-2Abohj8VY  start=30.0  end=40.0
[INFO] Downloading clip segment...

--- Processing m0vwa8RWzXg_260.000_270.000 ---
[INFO] yt_id=m0vwa8RWzXg  start=260.0  end=270.0
[INFO] Downloading clip segment...

--- Processing KIZay1zgOiE_90.000_100.000 ---
[INFO] yt_id=KIZay1zgOiE  start=90.0  end=100.0
[INFO] Downloading clip segment...

--- Processing ItvWPMW7RWE_150.000_160.000 ---
[INFO] yt_id=ItvWPMW7RWE  start=150.0  end=160.0
[INFO] Downloading clip segment...


Batch 0 (threads):   0%|          | 0/1000 [00:00<?, ?it/s]

[OK] Downloaded to: /var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/valor_clip_twbmul2z/clip.mp4
[INFO] Extracting frames (adaptive)...
[OK] Extracted 3 frame(s)
[OK] Frame extracted
[INFO] Extracting audio wav...
[OK] Audio extracted: 320334 bytes
[INFO] Cleaning up temporary files...
[OK] Cleanup complete
[SUCCESS] x-2Abohj8VY_30.000_40.000 processed in 8.19s

--- Processing ILE12hEW5Ck_30.000_40.000 ---
[INFO] yt_id=ILE12hEW5Ck  start=30.0  end=40.0
[INFO] Downloading clip segment...
[OK] Downloaded to: /var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/valor_clip_b7uvbwm2/clip.mp4
[INFO] Extracting frames (adaptive)...
[OK] Extracted 3 frame(s)
[OK] Frame extracted
[INFO] Extracting audio wav...
[OK] Audio extracted: 320356 bytes
[INFO] Cleaning up temporary files...
[OK] Cleanup complete
[SUCCESS] ItvWPMW7RWE_150.000_160.000 processed in 8.40s

--- Processing 1zoqi-E5vFY_50.000_60.000 ---
[INFO] yt_id=1zoqi-E5vFY  start=50.0  end=60.0
[INFO] Downloading clip segment...
[OK] Downloa

[h264 @ 0x17bcb7330] mmco: unref short failure


[OK] Downloaded to: /var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/valor_clip_2y0twnzx/clip.mp4
[INFO] Extracting frames (adaptive)...
[OK] Extracted 3 frame(s)
[OK] Frame extracted
[INFO] Extracting audio wav...
[OK] Audio extracted: 320328 bytes
[INFO] Cleaning up temporary files...
[OK] Cleanup complete
[SUCCESS] ysNK5RVF3Zw_0.000_10.000 processed in 8.49s

--- Processing -E013cWxoMo_10.000_20.000 ---
[INFO] yt_id=-E013cWxoMo  start=10.0  end=20.0
[INFO] Downloading clip segment...
[OK] Downloaded to: /var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/valor_clip_xb09ioz4/clip.mp4
[INFO] Extracting frames (adaptive)...
[OK] Extracted 3 frame(s)
[OK] Frame extracted
[INFO] Extracting audio wav...
[OK] Audio extracted: 306954 bytes
[INFO] Cleaning up temporary files...
[OK] Cleanup complete
[SUCCESS] dH-FZyKTY3s_30.000_40.000 processed in 8.75s

--- Processing zo1D2kSxnxs_60.000_70.000 ---
[INFO] yt_id=zo1D2kSxnxs  start=60.0  end=70.0
[INFO] Downloading clip segment...
[ERROR] yt-dlp 

[h264 @ 0x319a091b0] mmco: unref short failure
[h264 @ 0x319a091b0] mmco: unref short failure


... (389 outputs left out)

[h264 @ 0x319a2fd60] mmco: unref short failure
[h264 @ 0x319a2fd60] mmco: unref short failure


[OK] Audio extracted: 320178 bytes
[INFO] Cleaning up temporary files...
[OK] Cleanup complete
[SUCCESS] hkr8KzMLwDA_500.000_510.000 processed in 10.18s

--- Processing 476vNb6thyM_30.000_40.000 ---
[INFO] yt_id=476vNb6thyM  start=30.0  end=40.0
[INFO] Downloading clip segment...
[OK] Downloaded to: /var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/valor_clip_67tik2p3/clip.mp4
[INFO] Extracting frames (adaptive)...
[OK] Extracted 3 frame(s)
[OK] Frame extracted
[INFO] Extracting audio wav...
[OK] Audio extracted: 320334 bytes
[INFO] Cleaning up temporary files...
[OK] Cleanup complete
[SUCCESS] DJoFVhjQeDE_30.000_40.000 processed in 10.39s

--- Processing gdSmxqu5_UM_30.000_40.000 ---
[INFO] yt_id=gdSmxqu5_UM  start=30.0  end=40.0
[INFO] Downloading clip segment...
[OK] Downloaded to: /var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/valor_clip_4hcgqksn/clip.mp4
[INFO] Extracting frames (adaptive)...
[OK] Extracted 3 frame(s)
[OK] Frame extracted
[INFO] Extracting audio wav...
[OK] Downl

[h264 @ 0x11466db90] mmco: unref short failure
[h264 @ 0x11466db90] mmco: unref short failure


[OK] Downloaded to: /var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/valor_clip_a7un51w6/clip.mp4
[INFO] Extracting frames (adaptive)...
[OK] Extracted 3 frame(s)
[OK] Frame extracted
[INFO] Extracting audio wav...
[OK] Audio extracted: 320630 bytes
[INFO] Cleaning up temporary files...
[OK] Cleanup complete
[SUCCESS] XJXn88r9ys8_280.000_290.000 processed in 8.81s

--- Processing XhAANob18_8_150.000_160.000 ---
[INFO] yt_id=XhAANob18_8  start=150.0  end=160.0
[INFO] Downloading clip segment...
[OK] Downloaded to: /var/folders/l8/nbcrmtqs2838_6yr979chd1m0000gn/T/valor_clip_cqcotx6h/clip.mp4
[INFO] Extracting frames (adaptive)...
[OK] Extracted 3 frame(s)
[OK] Frame extracted
[INFO] Extracting audio wav...
[OK] Audio extracted: 320334 bytes
[INFO] Cleaning up temporary files...
[OK] Cleanup complete
[SUCCESS] y8ppQIdnEbc_30.000_40.000 processed in 7.39s

--- Processing DP0ZCe7cIa0_40.000_50.000 ---
[INFO] yt_id=DP0ZCe7cIa0  start=40.0  end=50.0
[INFO] Downloading clip segment...
[OK] Dow